# iSCORS deliverable — condensation axes (tidy)

Loads the video **once** (streaming, RAM-capped) at the top, then three sections:
1. **Paper replication** — gamma, CV², slope-3 condensation, y-intercept (+ optional Pearson vs the .mat Cond_map).
2. **Single-cell most-reproducible axis** — reliability-max (cross-half GEVD) and the X⊥Y residual.
3. **Multi-axis decomposition** — G(τ) GEVD axis 1/2 and de-nuisanced ICA comp 1/2.

Conventions (so results are reproducible):
- **ICA order/sign are arbitrary in FastICA** → fixed: sparse (high |kurtosis|) = comp 1, heavy tail positive.
- **De-nuisance basis = poly3 + radial vignetting**, identical wherever ICA is used.
- The temporal flat-field cancels in C(τ)/C(0), so g_norm is insensitive to it.


In [ ]:
# ── setup: imports + repo path ──
import os, gc, sys, zipfile, numpy as np, matplotlib.pyplot as plt, tifffile
from scipy.ndimage import zoom
from scipy.linalg import eigh as geigh
from scipy.stats import pearsonr, kurtosis
from sklearn.decomposition import FastICA
REPO = '/content/iscors-net'
if os.path.isdir(REPO):
    os.chdir(REPO)
    if REPO not in sys.path: sys.path.insert(0, REPO)
from utils.gpu_iscors_fit import streaming_acf, fit_gamma_alpha_batched


In [ ]:
# ── config (EDIT paths; same as the runner notebooks) ──
ZIP_PATH    = '/content/drive/MyDrive/iscors_test/large_file.zip'   # .zip or a .tif
VIDEO_FNAME = 'COBRI_rarw_video.tif'
EXTRACT_DIR = '/content/real_data'
MASK_PATH   = 'data/condensation_mask.tif'
MAT_FNAME   = 'Output_iSCORS_map.mat'        # optional, for the Cond_map validation
N_FRAMES = 5000; BIN = 2; SLOPE = 3.0; GAMMA_SCALE = 2.0
RECON_TAUS = (1, 2, 4, 8, 16, 32, 48, 64, 96, 128)

def resolve_video():
    p = ZIP_PATH
    if not str(p).lower().endswith('.zip'): return p
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    find = lambda: next((os.path.join(dp, VIDEO_FNAME) for dp, _, fs in os.walk(EXTRACT_DIR) if VIDEO_FNAME in fs), None)
    vp = find()
    if not vp:
        with zipfile.ZipFile(p) as z: z.extractall(EXTRACT_DIR)
        vp = find()
    assert vp, f'{VIDEO_FNAME} not found under {EXTRACT_DIR} (from {ZIP_PATH})'
    return vp

def load_mask(shape):
    mk = np.asarray(tifffile.imread(MASK_PATH)).squeeze()
    if mk.ndim == 3: mk = mk[..., 0]
    if mk.shape != shape: mk = zoom(mk, (shape[0]/mk.shape[0], shape[1]/mk.shape[1]), order=0)
    bd = np.zeros(shape, bool); bd[0,:]=bd[-1,:]=bd[:,0]=bd[:,-1]=True
    return mk == min(np.unique(mk), key=lambda z: (mk[bd] == z).mean())   # interior level = nucleus


In [ ]:
# ── load ONCE (streaming, RAM-capped): all per-pixel maps for full + halves, then done ──
VPATH = resolve_video()
def _maps(start, n):
    g, cell, c0, mean = streaming_acf(VPATH, RECON_TAUS, n_frames=n, bin_factor=BIN, start=start, chunk=200)
    fit = fit_gamma_alpha_batched(g, cell, RECON_TAUS, global_alpha=True,
                                  gamma_scale=GAMMA_SCALE, n_steps=400, verbose=False)
    return fit['gamma'].astype(np.float32), (c0 / (mean ** 2 + 1e-10)).astype(np.float32), g.astype(np.float32)
h = N_FRAMES // 2
gamma, dens, G = _maps(0, N_FRAMES)        # full
g1g, g1d, G1   = _maps(0, h)               # first half
g2g, g2d, G2   = _maps(h, h)               # second half
nuc = load_mask(dens.shape) & np.isfinite(gamma)
_lg = lambda a: np.log10(np.clip(a, 1e-12, None))
X,  Y  = _lg(1/np.clip(gamma, 1e-6, None)), _lg(dens)
X1, Y1 = _lg(1/np.clip(g1g, 1e-6, None)), _lg(g1d)
X2, Y2 = _lg(1/np.clip(g2g, 1e-6, None)), _lg(g2d)
gc.collect()
print('loaded once (video freed). nucleus px:', int(nuc.sum()))


In [ ]:
# ── helpers (also register maps for the viewer) ──
m = nuc & np.isfinite(X) & np.isfinite(Y)
ALL_MAPS = {}
def _full(vec_on_m):
    z = np.full(nuc.shape, np.nan); z[m] = vec_on_m; return z
def _gevd(a1, a2):
    Cg = (a1.T @ a2) / a1.shape[0]; Cg = (Cg + Cg.T) / 2
    v, w = geigh(Cg, np.cov(np.vstack([a1, a2]).T)); o = np.argsort(v)[::-1]
    return v[o], w[:, o]
def show(items, title):
    n = len(items); fig, ax = plt.subplots(1, n, figsize=(4.2*n, 4.2)); ax = np.atleast_1d(ax)
    for a, (t, d, cm) in zip(ax, items):
        ALL_MAPS[t] = (d, cm)
        im = a.imshow(d, cmap=cm, vmin=np.nanpercentile(d, 2), vmax=np.nanpercentile(d, 98))
        a.set_title(t, fontsize=10); a.axis('off'); plt.colorbar(im, ax=a, fraction=0.046)
    fig.suptitle(title); plt.tight_layout(); plt.show()


In [ ]:
# ── Section 1 — paper replication ──
b3 = float((Y[m] - SLOPE * X[m]).mean())
cond = _full(((X + SLOPE * Y - SLOPE * b3) / (1 + SLOPE ** 2))[m])
show([('gamma (log)',                _full(_lg(np.clip(gamma, 1e-6, None))[m]), 'inferno'),
      ('CV2 (log)',                  _full(Y[m]),                               'inferno'),
      ('slope-3 condensation',       cond,                                      'inferno'),
      ('y-intercept density (Y-3X)', _full((Y - SLOPE * X)[m]),                 'inferno')],
     'Section 1 - paper replication')

# optional: Pearson of our slope-3 condensation vs the .mat Cond_map
def _find(name):
    for dp, _, fs in os.walk(EXTRACT_DIR):
        if name in fs: return os.path.join(dp, name)
mp = _find(MAT_FNAME)
if mp:
    try:
        import scipy.io; mat = scipy.io.loadmat(mp)
    except NotImplementedError:
        import h5py; mat = {k: np.array(v) for k, v in h5py.File(mp, 'r').items()}
    if 'Cond_map' in mat:
        cg = np.asarray(mat['Cond_map']).astype(float).squeeze()
        o = np.rot90(np.flipud(cg), -1)                     # MATLAB → Python orientation
        if o.shape != nuc.shape: o = zoom(o, (nuc.shape[0]/o.shape[0], nuc.shape[1]/o.shape[1]), order=1)
        ok = nuc & np.isfinite(o) & np.isfinite(cond)
        print('Pearson(our slope-3 condensation, .mat Cond_map) =', round(pearsonr(o[ok], cond[ok])[0], 3))
else:
    print('(.mat Cond_map not found under EXTRACT_DIR — skipping validation)')


In [ ]:
# ── Section 2 — single-cell most-reproducible axis ──
mC = nuc & np.isfinite(X1) & np.isfinite(Y1) & np.isfinite(X2) & np.isfinite(Y2)
mx, my, sx, sy = X[mC].mean(), Y[mC].mean(), X[mC].std(), Y[mC].std()
f1 = np.vstack([(X1[mC]-mx)/sx, (Y1[mC]-my)/sy]); f2 = np.vstack([(X2[mC]-mx)/sx, (Y2[mC]-my)/sy])
_, Wr = _gevd(f1, f2); vrel = Wr[:, 0]
relmax = _full((vrel[0]*(X-mx)/sx + vrel[1]*(Y-my)/sy)[m])
def _rel(vec):
    p1 = vec[0]*(X1[mC]-mx)/sx + vec[1]*(Y1[mC]-my)/sy; p2 = vec[0]*(X2[mC]-mx)/sx + vec[1]*(Y2[mC]-my)/sy
    return pearsonr(p1, p2)[0]
a, b = np.polyfit(Y[m], X[m], 1); xperp = _full((X-(a*Y+b))[m])
print('reliability-max axis r(halves) =', round(_rel(vrel), 3), '  raw CV2 r(halves) =', round(pearsonr(Y1[mC], Y2[mC])[0], 3))
show([('reliability-max axis', relmax, 'inferno'), ('X-perp-Y residual', xperp, 'coolwarm')],
     'Section 2 - single-cell reproducible axis')


In [ ]:
# ── Section 3 — multi-axis decomposition (G(tau) GEVD + de-nuisanced ICA) ──
feat = lambda g, yl: np.concatenate([g, yl[..., None]], -1)
F1, F2, Ff = feat(G1, Y1)[m], feat(G2, Y2)[m], feat(G, Y)[m]
mu = (F1.mean(0)+F2.mean(0))/2; sd = (F1.std(0)+F2.std(0))/2 + 1e-9
_, W = _gevd((F1-mu)/sd, (F2-mu)/sd); pj = ((Ff-mu)/sd) @ W
yy, xx = np.mgrid[0:nuc.shape[0], 0:nuc.shape[1]]
xa = (xx[m]-xx[m].mean())/xx[m].std(); ya = (yy[m]-yy[m].mean())/yy[m].std()
D = np.vstack([np.ones_like(xa), xa, ya, xa**2, ya**2, xa*ya, xa**3, ya**3, xa**2*ya, xa*ya**2,
               np.sqrt(xa**2+ya**2), xa**2+ya**2]).T              # poly3 + radial vignetting
Pinv = np.linalg.pinv(D); dn = lambda Fm: Fm - D @ (Pinv @ Fm)
_, W2 = _gevd((dn(F1)-mu)/sd, (dn(F2)-mu)/sd)
S = FastICA(2, random_state=0, whiten='unit-variance', max_iter=1000).fit_transform(((dn(Ff)-mu)/sd) @ W2[:, :2])
S = S[:, np.argsort([-abs(kurtosis(S[:, 0])), -abs(kurtosis(S[:, 1]))])]   # sparse component first
for j in range(2):
    if abs(S[:, j].min()) > abs(S[:, j].max()): S[:, j] = -S[:, j]         # heavy tail positive
show([('G(tau) GEVD axis 1', _full(pj[:, 0]), 'inferno'), ('G(tau) GEVD axis 2', _full(pj[:, 1]), 'inferno'),
      ('ICA comp 1 (condensate)', _full(S[:, 0]), 'inferno'), ('ICA comp 2', _full(S[:, 1]), 'inferno')],
     'Section 3 - multi-axis decomposition')


In [ ]:
# ── viewer: pick any computed map (no video reload) ──
import gradio as gr
def _render(name):
    d, cm = ALL_MAPS[name]
    fig, a = plt.subplots(figsize=(5, 5))
    im = a.imshow(d, cmap=cm, vmin=np.nanpercentile(d, 2), vmax=np.nanpercentile(d, 98))
    a.axis('off'); plt.colorbar(im, ax=a, fraction=0.046); return fig
with gr.Blocks() as demo:
    gr.Markdown('## iSCORS axes viewer (cached maps)')
    names = list(ALL_MAPS)
    dd = gr.Dropdown(names, value=names[0], label='map')
    out = gr.Plot()
    dd.change(_render, dd, out); demo.load(lambda: _render(names[0]), None, out)
demo.launch(share=False)
